# Road Accidents Prediction - CatBoost Optimized
## Pure CatBoost Pipeline with Native Categorical Feature Handling

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

## 1. DATA LOADING

In [ ]:
# Load datasets
TRAIN_PATH = r'C:\Users\ASUS\Desktop\Sem04\Coding\sem04_Codes\UOM_sem_04\Inputs\Accident\train.csv'
TEST_PATH = r'C:\Users\ASUS\Desktop\Sem04\Coding\sem04_Codes\UOM_sem_04\Inputs\Accident\test.csv'
TARGET_COL = 'accident_risk'
ID_COL = 'id'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")

## 2. DATA PREPROCESSING

In [ ]:
# Separate features and target
X = train_df.drop(columns=[TARGET_COL, ID_COL])
y = train_df[TARGET_COL]

# Save test IDs for submission
test_ids = test_df[ID_COL].copy()
X_test = test_df.drop(columns=[ID_COL])

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Drop unnecessary columns
drop_cols = ["road_signs_present", "school_season", "num_lanes", "time_of_day"]

X = X.drop(columns=drop_cols, errors='ignore')
X_test = X_test.drop(columns=drop_cols, errors='ignore')

print(f"Features after dropping: {X.columns.tolist()}")

In [ ]:
# Identify numeric and categorical columns
# CatBoost handles categorical features natively!
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Numeric columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")
print(f"\nCatBoost will handle these categorical features natively (no encoding needed!)")

In [ ]:
# Impute missing values
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

if num_cols:
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
    X_test[num_cols] = num_imputer.transform(X_test[num_cols])

if cat_cols:
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
    X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

print("Missing values imputed")
print(f"Missing in X: {X.isnull().sum().sum()}")
print(f"Missing in X_test: {X_test.isnull().sum().sum()}")

In [ ]:
# Convert categorical columns to string type for CatBoost
# This helps CatBoost identify them as categorical features
for col in cat_cols:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f"Categorical features converted to string type")
print(f"\nData types:")
print(X.dtypes)

In [ ]:
# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

## 3. CATBOOST BASELINE MODEL

In [ ]:
# Train baseline CatBoost model
baseline_model = CatBoostRegressor(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
    cat_features=cat_cols,
    verbose=False
)

print("Training baseline CatBoost model...")
baseline_model.fit(X_train, y_train)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_val)
baseline_rmse = np.sqrt(mean_squared_error(y_val, y_pred_baseline))
baseline_r2 = r2_score(y_val, y_pred_baseline)
baseline_mae = mean_absolute_error(y_val, y_pred_baseline)

print(f"\nBaseline CatBoost Performance:")
print(f"  RMSE: {baseline_rmse:.6f}")
print(f"  R²:   {baseline_r2:.6f}")
print(f"  MAE:  {baseline_mae:.6f}")

## 4. HYPERPARAMETER TUNING WITH RANDOMIZEDSEARCHCV

In [ ]:
# Define hyperparameter search space for CatBoost
param_grid = {
    'iterations': [100, 200, 300, 500],
    'depth': [4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'l2_leaf_reg': [1, 3, 5, 10, 20],
    'min_data_in_leaf': [1, 5, 10],
    'max_bin': [32, 64, 128]
}

print("Starting RandomizedSearchCV for hyperparameter tuning...")
print("This may take a few minutes...\n")

catboost_search = RandomizedSearchCV(
    estimator=CatBoostRegressor(
        random_state=42,
        cat_features=cat_cols,
        verbose=False
    ),
    param_distributions=param_grid,
    n_iter=40,  # Try 40 combinations
    scoring='r2',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

catboost_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for param, value in catboost_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV R² Score: {catboost_search.best_score_:.6f}")

## 5. TRAIN OPTIMIZED MODEL

In [ ]:
# Create optimized model with best parameters
optimized_model = CatBoostRegressor(
    **catboost_search.best_params_,
    random_state=42,
    cat_features=cat_cols,
    verbose=False
)

print("Training optimized CatBoost model...")
optimized_model.fit(X_train, y_train)

# Evaluate on validation set
y_pred_optimized = optimized_model.predict(X_val)
opt_rmse = np.sqrt(mean_squared_error(y_val, y_pred_optimized))
opt_r2 = r2_score(y_val, y_pred_optimized)
opt_mae = mean_absolute_error(y_val, y_pred_optimized)

print(f"\nOptimized CatBoost Performance (Validation Set):")
print(f"  RMSE: {opt_rmse:.6f}")
print(f"  R²:   {opt_r2:.6f}")
print(f"  MAE:  {opt_mae:.6f}")

# Compare with baseline
rmse_improvement = ((baseline_rmse - opt_rmse) / baseline_rmse) * 100
r2_improvement = ((opt_r2 - baseline_r2) / baseline_r2) * 100

print(f"\nImprovement over Baseline:")
print(f"  RMSE: {rmse_improvement:.2f}% better")
print(f"  R²:   {r2_improvement:.2f}% better")

## 6. CROSS-VALIDATION ON FULL TRAINING DATA

In [ ]:
# Cross-validation scores on full training data
cv_scores = cross_val_score(
    optimized_model, X, y, cv=5, scoring='r2', n_jobs=-1
)

print(f"Cross-Validation Scores (5-fold):")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: {score:.6f}")

print(f"\nMean CV R² Score: {cv_scores.mean():.6f} (+/- {cv_scores.std():.6f})")

## 7. FEATURE IMPORTANCE ANALYSIS

In [ ]:
# Get feature importances (CatBoost uses Shap values by default)
feature_importance = pd.Series(
    optimized_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

print(f"\nTop 15 Most Important Features:")
print(feature_importance.head(15))

# Percentage importance
total_importance = feature_importance.sum()
cumulative_importance = feature_importance.cumsum() / total_importance

print(f"\nTop features cover:")
for n_features in [5, 10, 15]:
    if n_features <= len(cumulative_importance):
        coverage = cumulative_importance.iloc[n_features-1] * 100
        print(f"  Top {n_features} features: {coverage:.2f}% of model importance")

## 8. FINAL PREDICTIONS ON TEST SET

In [ ]:
# Retrain on full dataset for maximum accuracy
print("Retraining on full dataset...")
final_model = CatBoostRegressor(
    **catboost_search.best_params_,
    random_state=42,
    cat_features=cat_cols,
    verbose=False
)

final_model.fit(X, y)

# Generate predictions on test set
print("Generating predictions on test set...")
test_predictions = final_model.predict(X_test)

print(f"\nTest predictions generated!")
print(f"Predictions shape: {test_predictions.shape}")
print(f"Predictions range: [{test_predictions.min():.6f}, {test_predictions.max():.6f}]")
print(f"Predictions mean: {test_predictions.mean():.6f}")

## 9. GENERATE SUBMISSION FILE

In [ ]:
from pathlib import Path

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'accident_risk': test_predictions
})

# Save to outputs folder
output_path = Path('./Outputs/submission_catboost_optimized.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

submission.to_csv(output_path, index=False)

print(f"✓ Submission saved to {output_path}")
print(f"\nSubmission Preview (first 10 rows):")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")

## 10. MODEL SUMMARY

In [ ]:
print("="*60)
print("CATBOOST OPTIMIZED MODEL SUMMARY")
print("="*60)
print(f"\nOptimized Hyperparameters:")
for param, value in catboost_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nPerformance Metrics (Validation Set):")
print(f"  RMSE:        {opt_rmse:.6f}")
print(f"  R² Score:    {opt_r2:.6f}")
print(f"  MAE:         {opt_mae:.6f}")

print(f"\nCross-Validation (5-fold):")
print(f"  Mean R²:     {cv_scores.mean():.6f}")
print(f"  Std Dev:     {cv_scores.std():.6f}")

print(f"\nMost Important Feature: {feature_importance.index[0]}")
print(f"Feature Importance: {feature_importance.iloc[0]:.6f}")

print(f"\nDataset Information:")
print(f"  Training samples: {X_train.shape[0]}")
print(f"  Validation samples: {X_val.shape[0]}")
print(f"  Test samples: {X_test.shape[0]}")
print(f"  Total features: {X.shape[1]}")
print(f"  Categorical features: {len(cat_cols)}")
print(f"  Numeric features: {len(num_cols)}")

print(f"\nKey CatBoost Advantages:")
print(f"  ✓ Native categorical feature handling")
print(f"  ✓ No encoding needed for categorical variables")
print(f"  ✓ Built-in SHAP value computation")
print(f"  ✓ Ordered boosting for better generalization")

print("="*60)

## 11. COMPARISON WITH OTHER MODELS (Optional)

In [ ]:
# If you want to compare CatBoost with XGBoost on this dataset
# Uncomment below and run

# from xgboost import XGBRegressor
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline

# # XGBoost requires one-hot encoding
# X_train_xgb = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
# X_val_xgb = pd.get_dummies(X_val, columns=cat_cols, drop_first=False)
# X_train_xgb, X_val_xgb = X_train_xgb.align(X_val_xgb, join='left', axis=1, fill_value=0)

# xgb_model = XGBRegressor(
#     n_estimators=500,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.7,
#     colsample_bytree=0.7,
#     random_state=42,
#     n_jobs=-1,
#     verbosity=0
# )

# xgb_model.fit(X_train_xgb, y_train)
# y_pred_xgb = xgb_model.predict(X_val_xgb)
# xgb_r2 = r2_score(y_val, y_pred_xgb)

# print(f"CatBoost R²: {opt_r2:.6f}")
# print(f"XGBoost R²:  {xgb_r2:.6f}")
# print(f"Winner: {'CatBoost' if opt_r2 > xgb_r2 else 'XGBoost'}")